# Import data

In [1]:
import pandas as pd
df = pd.read_csv('../data/raw/voc-data.csv', encoding='utf-8', on_bad_lines='skip')
df.head()

,หมายเลขเสียง,ประเภทสารสนเทศ,วันที่บันทึกข้อมูล,พื้นที่ที่มีประเด็นเสียง,Unnamed: 4,Unnamed: 5,ช่องทางการรับฟังเสียง,กลไก,CA,ชื่อลูกค้า,กลุ่มลูกค้า,พื้นที่จ่ายไฟ,วันที่ได้รับข้อมูล,หัวข้อเสียง,ประเด็นเสียง,ประเภท VOC,รายละเอียดเสียง
0,NaN,NaN,NaN,สายงาน,ฝ่าย/เขต,กอง/การไฟฟ้า,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,C-59720851,ความคาดหวัง,29/03/2024 10:12 น.,สายงานการไฟฟ้า ภาค 2,การไฟฟ้าเขต 2 (อุบลราชธานี) ภาค 2,E09201 : การไฟฟ้าส่วนภูมิภาคสาขาอำเภอโพนทอง จั...,Physical,1129 PEA Contact Center,NaN,สุรพล ผงทอง,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,\nการให้บริการ - ทั่วไป,ติดตามสถานะการขอใช้บริการ\n,บริการ,ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะกา...
2,M-67002090,ความต้องการ,29/03/2024 10:12 น.,NaN,NaN,NaN,Digital,PEA Mobile Application (Smart Plus),020012705638,ประภาพร แพงถิ่น,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,ความถูกต้องของข้อมูลบน Website/Application PEA,สนับสนุน,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...
3,C-59721018,ความคาดหวัง,29/03/2024 10:09 น.,ฝ่ายงานผู้ว่าการ,NaN,NaN,Physical,1129 PEA Contact Center,NaN,วีราพล N/A,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,การใช้งานบน Website/Application PEA,สนับสนุน,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: ยกเลิกเบ...
4,C-59720995,ความคาดหวัง,29/03/2024 10:07 น.,ฝ่ายงานผู้ว่าการ,NaN,NaN,Physical,1129 PEA Contact Center,NaN,ดวงใจ N/A,ลูกค้ารายย่อย / บ้านอยู่อาศัย,NaN,29/03/2024,การให้บริการ - ระบบออนไลน์,การใช้งานบน Website/Application PEA,สนับสนุน,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...


In [2]:
df.dropna(subset=['รายละเอียดเสียง'], inplace=True)
text_df = df[['หมายเลขเสียง','รายละเอียดเสียง']].copy()

In [3]:
text_df.rename(columns={'หมายเลขเสียง':'voc_id','รายละเอียดเสียง': 'text'}, inplace=True)

In [4]:
text_df.head()

,voc_id,text
1,C-59720851,ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะกา...
2,M-67002090,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...
3,C-59721018,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: ยกเลิกเบ...
4,C-59720995,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...
5,C-59720934,ร้องขอกฟอ สามชุก จังหวัดสุพรรณบุรี เรื่องสถานะ...


# Define function to doing text nane entity recognition

In [5]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from pythainlp.tokenize import word_tokenize # pip install pythainlp
import torch


In [6]:
text_df.loc[1]['text']

'ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะการขอขยายเขต ของบ้านเลขที่167 ม.13 ต.สระนกแก้ว อ.โพนทอง จ.ร้อยเอ็ด ชื่อผู้ขอจดทะเบียนใช้ไฟฟ้าทองอินทร์ ผงทอง วันที่ยื่นเรื่อง5/3/2567 ผู้ใช้ไฟฟ้าแจ้งว่าได้ไปส่งเอกสารครบทั้งหมดแล้วเจ้าหน้าที่แจ้งว่างบประมาณหมด ต้องส่งเรื่องไปการไฟฟ้าอุบลราชธานี แต่เจ้าหน้าที่ยังไม่มีการส่งเรื่องไป ผชฟ.ต้องให้การไฟฟ้าโฟนทองส่งเรื่องไปที่การไฟฟ้าอุบลราชธานีให้ เนื่องจากต้องการใช้ไฟฟ้าด่วน รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //คุณสุรพล  หมายเลขติดต่อกลับ 0986451488'

In [ ]:
# Load model/tokenizer ONCE outside the function
name = "Porameht/wangchanberta-thainer-corpus-v2-2"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForTokenClassification.from_pretrained(name)

def get_ner_tag(sentence, tokenizer, model, window=200, stride=100):
    """
    ใช้ sliding window with overlap เพื่อรองรับข้อความยาวที่เกิน max_length
    - window: จำนวน word ต่อ chunk
    - stride: จำนวน word ที่เลื่อนต่อรอบ (overlap = window - stride)
    """
    cut = word_tokenize(sentence.replace(" ", "<_>"))
    final_word_tags = {}  # global word index -> tag

    start = 0
    while start < len(cut):
        end = min(start + window, len(cut))
        chunk_words = cut[start:end]

        chunk_inputs = tokenizer(
            chunk_words,
            is_split_into_words=True,
            return_tensors="pt",
            max_length=512,
            truncation=True,
        )

        ids = chunk_inputs["input_ids"]
        mask = chunk_inputs["attention_mask"]
        outputs = model(ids, attention_mask=mask)
        predictions = torch.argmax(outputs[0], dim=2)
        predicted_token_class = [model.config.id2label[t.item()] for t in predictions[0]]

        # Map subword tokens → original word ผ่าน word_ids() แทน decode()
        word_ids = chunk_inputs.word_ids(batch_index=0)
        seen = set()
        for token_pos, word_idx in enumerate(word_ids):
            if word_idx is None:
                continue
            if word_idx in seen:
                continue  # ใช้ subword แรกของแต่ละ word
            seen.add(word_idx)
            global_idx = start + word_idx
            if global_idx not in final_word_tags:
                final_word_tags[global_idx] = predicted_token_class[token_pos]

        if end >= len(cut):
            break
        start += stride

    # Reconstruct (word, tag) list จาก word ต้นฉบับ
    result = []
    for idx, word in enumerate(cut):
        tag = final_word_tags.get(idx, 'O')
        if word == "<_>":
            word = " "
        result.append((word, tag))

    return result


In [30]:
text = text_df.loc[1]['text']
ner_tag = get_ner_tag(text, tokenizer, model)

In [31]:
ner_tag

[('ร้อง', 'O'),
 ('ขอ', 'O'),
 ('ก', 'O'),
 ('ฟ', 'O'),
 ('อ', 'I-ORGANIZATION'),
 ('.', 'B-LOCATION'),
 ('โพน', 'I-ORGANIZATION'),
 ('ทอง', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('จังหวัด', 'B-LOCATION'),
 ('ร้อยเอ็ด', 'I-LOCATION'),
 (' ', 'O'),
 ('เรื่อง', 'O'),
 ('สถานะ', 'O'),
 ('การ', 'O'),
 ('ขอ', 'O'),
 ('ขยาย', 'O'),
 ('เขต', 'O'),
 (' ', 'O'),
 ('ของ', 'O'),
 ('บ้าน', 'O'),
 ('เลขที่', 'I-LOCATION'),
 ('16', 'I-LOCATION'),
 ('7', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('ม', 'I-LOCATION'),
 ('.', 'I-LOCATION'),
 ('13', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('ต', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('สระ', 'I-LOCATION'),
 ('นก', 'I-LOCATION'),
 ('แก้ว', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('อ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('โพน', 'I-LOCATION'),
 ('ทอง', 'I-LOCATION'),
 (' ', 'B-LOCATION'),
 ('จ', 'B-LOCATION'),
 ('.', 'I-LOCATION'),
 ('ร้อยเอ็ด', 'I-LOCATION'),
 (' ', 'O'),
 ('ชื่อ', 'O'),
 ('ผู้', 'O'),
 ('ขอ', 'O'),
 ('จดทะเบียน', 'O'),
 ('ใช้', 'O'),
 ('ไฟฟ้า', 

In [32]:
def ner_nasking(ner_tag:list):
    masked = ""
    current_entity = None 

    for token, tag in ner_tag:
        if tag == 'O':
            masked += token
            current_entity = None
        else:
            entity = tag.split('-')[1]
            if tag.startswith('B-') or current_entity != entity:
                masked += f"[{entity}]"
                current_entity = entity
            elif tag.startswith('I-') and current_entity == entity:
                continue
            else:
                current_entity = None
                masked += token

    return masked.strip()

marked_text = ner_nasking(ner_tag)
print(marked_text)

ร้องขอกฟ[ORGANIZATION][LOCATION][ORGANIZATION][LOCATION][LOCATION][LOCATION] เรื่องสถานะการขอขยายเขต ของบ้าน[LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION][LOCATION] ชื่อผู้ขอจดทะเบียนใช้ไฟฟ้า[PERSON]อินทร์ ผงทอง วันที่ยื่นเรื่อง[DATE]67 ผู้ใช้ไฟฟ้าแจ้งว่าได้ไปส่งเอกสารครบทั้งหมดแล้วเจ้าหน้าที่แจ้งว่างบประมาณหมด ต้องส่งเรื่องไป[ORGANIZATION] แต่เจ้าหน้าที่ยังไม่มีการส่งเรื่องไป [ORGANIZATION].ต้องให้[ORGANIZATION]ส่งเรื่องไปที่[ORGANIZATION]ให้ เนื่องจากต้องการใช้ไฟฟ้าด่วน รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //[PERSON]  หมายเลขติดต่อกลับ [PHONE]88


In [33]:
import re

def remove_redundant_tags(text):
    # ลบ tag ซ้ำติดกัน เช่น [ORG][ORG] เหลือ [ORG]
    return re.sub(r'(\[[A-Z]+\])\1+', r'\1', text)

In [34]:
text_ner = text_df.copy()

In [ ]:
for i, row in text_ner.iterrows():
    try:
        ner_tag = get_ner_tag(row['text'], tokenizer, model)
        marked_text = ner_nasking(ner_tag)
        cleaned_text = remove_redundant_tags(marked_text)
        text_ner.at[i, 'marked_text'] = cleaned_text
    except Exception as e:
        print(f"[SKIP] Row {i}: {e}")
        continue


[SKIP] Row 237: index out of range in self
[SKIP] Row 246: index out of range in self
[SKIP] Row 255: index out of range in self
[SKIP] Row 594: index out of range in self
[SKIP] Row 842: index out of range in self
[SKIP] Row 866: index out of range in self
[SKIP] Row 881: index out of range in self
[SKIP] Row 1705: index out of range in self
[SKIP] Row 2698: index out of range in self
[SKIP] Row 3142: index out of range in self
[SKIP] Row 3307: index out of range in self
[SKIP] Row 4035: index out of range in self
[SKIP] Row 4776: index out of range in self
[SKIP] Row 4779: index out of range in self
[SKIP] Row 4801: index out of range in self
[SKIP] Row 4802: index out of range in self
[SKIP] Row 4996: index out of range in self
[SKIP] Row 5013: index out of range in self
[SKIP] Row 5024: index out of range in self
[SKIP] Row 5432: index out of range in self
[SKIP] Row 5487: index out of range in self
[SKIP] Row 5575: index out of range in self
[SKIP] Row 5673: index out of range in 

In [36]:
text_ner

,voc_id,text,marked_text
1,C-59720851,ร้องขอกฟอ.โพนทอง จังหวัดร้อยเอ็ด เรื่องสถานะกา...,ร้องขอกฟ[ORGANIZATION][LOCATION][ORGANIZATION]...
2,M-67002090,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...,อยู่บ้านเลขที่ที่ 118/8 ค่ะ แต่พอเช็คในแอฟกับไ...
3,C-59721018,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: ยกเลิกเบ...,แจ้งปัญหา <unk> <unk>mart <unk>lusปัญหาที่พบ: ...
4,C-59720995,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...,แจ้งปัญหา <unk> <unk>mart <unk>lusปัญหาที่พบ: ...
5,C-59720934,ร้องขอกฟอ สามชุก จังหวัดสุพรรณบุรี เรื่องสถานะ...,ร้อง[ORGANIZATION][LOCATION] เรื่องสถานะการขอใ...
...,...,...,...
19749,C-58943347,แนะนำ กฟจ.ประจวบคีรีขันธ์ จังหวัด ประจวบคีรีขั...,แนะนํา[ORGANIZATION] [LOCATION]คีรีขันธ์ เรื่อ...
19750,I-67000001,มี​ SMS​แจ้งให้รับคืนเงินประกันมิเตอร์ผ่านหมาย...,มี[ORGANIZATION]<unk>แจ้งให้รับคืนเงินประกันมิ...
19751,C-58943585,แจ้งปัญหา PEA Smart Plus\nปัญหาที่พบ: เปลี่ยนเ...,แจ้งปัญหา <unk> <unk>mart <unk>lusปัญหาที่พบ: ...
19752,C-58943308,แจ้งปัญหา > บริการอื่น ๆ > บริการยานยนต์ไฟฟ้า ...,แจ้งปัญหา > บริการอื่น ๆ > บริการยานยนต์ไฟฟ้า ...


In [37]:
text_ner.loc[9, 'marked_text']

'ร้องขอ[ORGANIZATION]ฟ[ORGANIZATION][LOCATION]  เรื่องสถานะการขอใช้ไฟฟ้าเครื่องที่ 2  ของบ้านเลขที่ 45[LOCATION]63(<unk>)[LOCATION]  หมายเลขผู้ใช้ไฟฟ้า [PHONE]89 ชื่อผู้จดทะเบียนใช้ไฟฟ้า[PERSON] วันที่ยื่นเรื่อง [DATE]67 มิเตอร์ขนาด 14 (45) แอมป์ ผู้ใช้ไฟฟ้าแจ้งว่าวันที่ [DATE]67 ได้ทําการติดต่อ[ORGANIZATION] ทาง[ORGANIZATION]แจ้งว่าจะเข้าตรวจสอบวันถัดไปเป็นวันที่ [DATE]67 จนถึงตอนนี้ก็ยังไม่มีเจ้าหน้าที่ติดต่อกลับ และยังไม่เข้าดําเนินการ  รบกวนหน่วยงานที่เกี่ยวข้องประสานงานตรวจสอบและติดต่อกลับผู้ใช้ไฟฟ้า //[PERSON]  หมายเลขติดต่อกลับ [PHONE]66'

In [38]:
text_ner.to_csv('../data/processed/voc_data_marked_text_pea.csv', index=False, encoding='utf-8')

In [39]:
text_ner.to_excel('../data/processed/voc_data_marked_text_pea.xlsx', index=False)